<a href="https://colab.research.google.com/github/MyTeachers123/agents-assignment-2/blob/main/assignment2_live_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 – Live Test in Colab
**Workspace Assistant (Google ADK): Google Calendar tools + GitHub MCP + tool search bonus**

This notebook runs the real agent from `MyTeachers123/agents-assignment-2` against **your real Google Calendar and GitHub**, and prints every tool call so you can see it working.

**Before you start – Colab secrets (🔑 icon on the left, turn *Notebook access* ON):**
| Secret name | Value |
|---|---|
| `GOOGLE_API_KEY` | Gemini API key (same as the Week 2 lab) |
| `GITHUB_PERSONAL_ACCESS_TOKEN` | your `ghp_...` token |

You also need the `credentials.json` file (Google OAuth *Desktop app* client) on your computer.
Everything below runs on Colab's temporary machine, not on your computer.

**How to run:** run the cells **one by one from the top**. Cell 1 restarts Colab automatically at the end (you will see *"Your session crashed"* – that is expected). Then continue with cell 2.

## 1. Get the code, install, and restart (run once)

In [ ]:
import os
if not os.path.exists("/content/agents-assignment-2"):
    !git clone -q https://github.com/MyTeachers123/agents-assignment-2.git
%cd /content/agents-assignment-2/workspace_assistant
# -U upgrades Colab's pre-installed (older) mcp / google-adk
!pip install -q -U -r requirements.txt "mcp>=2" "google-adk>=2.11"
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs npm && node --version)
print("\n✅ Installed. Colab will now RESTART automatically (a 'session crashed' message is expected).")
print("➡️  After it restarts, continue with cell 2. Do NOT run cell 1 again.")
import time, sys; sys.stdout.flush(); time.sleep(3)
os.kill(os.getpid(), 9)   # restart Python so the upgraded packages are used

/content/agents-assignment-2/workspace_assistant
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 13.6 MB/s et

## 2. Load secrets and settings

In [18]:
%cd /content/agents-assignment-2/workspace_assistant
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"] = userdata.get("GITHUB_PERSONAL_ACCESS_TOKEN")
os.environ["GITHUB_USERNAME"] = "MyTeachers123"
os.environ["MODEL_NAME"] = "gemini-3.8-flash"
os.environ["CALENDAR_TIMEZONE"] = "America/Los_Angeles"
print("✅ Secrets loaded (values hidden)")

import importlib.metadata as md
from google.adk.tools.mcp_tool import McpToolset   # fails here if cell 1's restart did not happen
print("✅ mcp", md.version("mcp"), "| google-adk", md.version("google-adk"))

/content/agents-assignment-2/workspace_assistant
✅ Secrets loaded (values hidden)
✅ mcp 2.3.0 | google-adk 2.11.0


## 3. Upload `credentials.json` (Google OAuth Desktop client)

In [2]:
from google.colab import files
from pathlib import Path

cred_file = Path("config/credentials/credentials.json")
if cred_file.exists():
    print("✅ credentials.json already uploaded – skipping")
else:
    up = files.upload()                   # choose credentials.json
    cred_file.write_bytes(up[next(iter(up))])
    print("✅ Saved", cred_file)

Saving credentials.json to credentials.json
✅ Saved config/credentials/credentials.json


## 4. Sign in to Google Calendar (one time)
1. Run the cell and click the printed link. Sign in with the Gmail you added as a **Test user**.
2. If you see *Google hasn't verified this app* → **Continue**. Allow calendar access.
3. The browser then goes to a page that **fails to load** (`http://localhost/?state=...&code=...`). That is expected.
4. **Copy that whole address from the address bar** and paste it into the box below.

In [3]:
from urllib.parse import urlparse, parse_qs
from google_auth_oauthlib.flow import InstalledAppFlow
from tools.auth import SCOPES, CREDENTIALS_FILE, TOKEN_FILE

if TOKEN_FILE.exists():
    print("✅ Already signed in (token.json found) – skipping")
else:
  flow = InstalledAppFlow.from_client_secrets_file(str(CREDENTIALS_FILE), SCOPES["calendar"],
                                                   redirect_uri="http://localhost")
  auth_url, _ = flow.authorization_url(prompt="consent", access_type="offline")
  print("👉 Open this link:\n", auth_url)

  redirected = input("\nPaste the full http://localhost/... address here: ").strip()
  code = parse_qs(urlparse(redirected).query).get("code", [redirected])[0]
  flow.fetch_token(code=code)
  TOKEN_FILE.write_text(flow.credentials.to_json())
  print("✅ Signed in. token.json saved on this Colab machine only.")

👉 Open this link:
 https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=REDACTED_CLIENT_ID.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=REDACTED&code_challenge=hg4snCIJCPYCYB3B2xfvNcL8L75daed-RhFZKWYDMQ0&code_challenge_method=S256&prompt=consent&access_type=offline

Paste the full http://localhost/... address here: http://localhost/?state=REDACTED&iss=https://accounts.google.com&code=REDACTED&scope=https://www.googleapis.com/auth/calendar
✅ Signed in. token.json saved on this Colab machine only.


In [19]:
import os, json, tempfile
import tools.mcp_tools as mt

# Colab fix: send the GitHub MCP server's log to a file (Colab's stderr has no fileno)
_log = open(os.path.join(tempfile.gettempdir(), "github_mcp.log"), "a")
_orig = mt.GitHubMcpToolset.__init__
def _init(self, *a, **k):
    k.setdefault("errlog", _log); _orig(self, *a, **k)
mt.GitHubMcpToolset.__init__ = _init
mt.mcp_tools[:] = mt._build_mcp_tools()

from google.adk.models.llm_response import LlmResponse
from google.adk.runners import InMemoryRunner
from google.genai import types
from agent import create_agent, create_agent_with_tool_search
from tools.calendar_tools import calendar_tools
CAL = {f.__name__ for f in calendar_tools}

async def declared_tools(agent):
    """Capture exactly what would be sent to Gemini, then stop (no Gemini call, no quota used)."""
    seen = {}
    def capture(callback_context, llm_request):
        decls = [fd.model_dump(exclude_none=True, mode="json")
                 for t in (llm_request.config.tools or [])
                 for fd in (getattr(t, "function_declarations", None) or [])]
        seen["names"] = sorted(llm_request.tools_dict)
        seen["all"] = len(json.dumps(decls)) // 4
        seen["github"] = len(json.dumps([d for d in decls if d["name"] not in CAL])) // 4
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(text="measured")]))
    agent.before_model_callback = capture
    r = InMemoryRunner(agent=agent); r.auto_create_session = True
    async for _ in r.run_async(user_id="m", session_id="m", new_message=types.Content(
            role="user", parts=[types.Part(text="List my GitHub repositories")])):
        pass
    return seen

print("⏳ Starting the real GitHub MCP server (first time ~30s)…")
full = await declared_tools(create_agent())
lazy = await declared_tools(create_agent_with_tool_search())
pct = lambda a, b: f"{(1 - b / a) * 100:.0f}%"
print(f"\n{'':32}{'WITHOUT defer_loading':>24}{'WITH defer_loading':>22}{'Saved':>8}")
print(f"{'Tools sent to Gemini':32}{len(full['names']):>24}{len(lazy['names']):>22}{pct(len(full['names']), len(lazy['names'])):>8}")
print(f"{'GitHub tool tokens (~)':32}{full['github']:>24}{lazy['github']:>22}{pct(full['github'], lazy['github']):>8}")
print(f"{'All tool tokens (~)':32}{full['all']:>24}{lazy['all']:>22}{pct(full['all'], lazy['all']):>8}")
print("\nWITH defer_loading, GitHub tools loaded up front:", [n for n in lazy["names"] if n not in CAL])

print("\n🔎 search_github_tools loads tools only when needed:")
class _S: id = "demo"
class _TC:
    def __init__(self): self.state = {}; self.session = _S()
tc = _TC()
for q in ["create issue", "README file", "pull request"]:
    r = await mt.search_github_tools(q, tc)
    print(f"   '{q}' → {[t['name'] for t in r.get('loaded_tools', [])]}")

⏳ Starting the real GitHub MCP server (first time ~30s)…

                                   WITHOUT defer_loading    WITH defer_loading   Saved
Tools sent to Gemini                                  32                     9     72%
GitHub tool tokens (~)                              6478                   601     91%
All tool tokens (~)                                 8558                  2680     69%

WITH defer_loading, GitHub tools loaded up front: ['list_issues', 'search_github_tools', 'search_repositories']

🔎 search_github_tools loads tools only when needed:
   'create issue' → ['create_issue', 'add_issue_comment', 'create_branch']
   'README file' → ['create_or_update_file', 'get_file_contents', 'get_pull_request_files']
   'pull request' → ['create_pull_request', 'create_pull_request_review', 'get_pull_request']


## 5. Call the calendar tools directly (no LLM) – proves the Google API works

In [4]:
from tools.calendar_tools import get_current_datetime, list_upcoming_events
from datetime import date, timedelta

now = get_current_datetime()          # date in Los Angeles (Colab's clock is UTC)
print(now)
today = date.fromisoformat(now["date"])
week = list_upcoming_events(start_date=str(today), end_date=str(today + timedelta(days=7)))
print("status:", week["status"], "| events in the next 7 days:", week.get("count"))
for e in week.get("events", [])[:5]:
    print(" -", e["start"], "|", e["summary"])

{'status': 'success', 'now': '2026-10-11 22:48', 'date': '2026-10-11', 'weekday': 'Sunday', 'timezone': 'America/Los_Angeles'}
status: success | events in the next 7 days: 1
 - Fri Oct 16, 2026 2:00 PM | Personal appointment


## 6. Build the agent + a `chat()` helper that shows every tool call
`mode="full"` loads all GitHub tools; `mode="search"` uses the tool-search bonus (defer_loading).

In [5]:
import json, uuid
from google.adk.runners import InMemoryRunner
from google.genai import types
from agent import create_agent, create_agent_with_tool_search

runners, sessions = {}, {}

def _runner(mode):
    if mode not in runners:
        agent = create_agent_with_tool_search() if mode == "search" else create_agent()
        r = InMemoryRunner(agent=agent, app_name=f"wa_{mode}")
        r.auto_create_session = True
        runners[mode], sessions[mode] = r, uuid.uuid4().hex
    return runners[mode]

def _short(v, n=110):
    s = json.dumps(v, default=str, ensure_ascii=False)
    return s if len(s) <= n else s[:n] + "…"

async def chat(text, mode="full"):
    print(f"🧑 {text}")
    runner, tokens = _runner(mode), []
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    async for ev in runner.run_async(user_id="colab", session_id=sessions[mode], new_message=msg):
        u = getattr(ev, "usage_metadata", None)
        if u and u.prompt_token_count:
            tokens.append(u.prompt_token_count)
        for p in (ev.content.parts if ev.content and ev.content.parts else []):
            if p.function_call:
                print(f"   🔧 {p.function_call.name}({_short(dict(p.function_call.args or {}))})")
            elif p.function_response:
                r = p.function_response.response or {}
                status = r.get("status") or ("error" if r.get("isError") else "ok") if isinstance(r, dict) else "ok"
                extra = f" | {r.get('message')}" if isinstance(r, dict) and r.get("message") else ""
                print(f"   ↳ {p.function_response.name} → {status}{extra}")
            elif p.text and not getattr(p, "thought", False):
                print(f"🤖 {p.text.strip()}")
    print(f"   📊 prompt tokens per model call: {tokens}  (mode={mode})\n")

def new_chat(mode="full"):
    sessions[mode] = uuid.uuid4().hex
print("✅ Ready")

✅ Ready


## 7. Calendar tests

In [7]:
import sys, os, tempfile, asyncio, logging
import tools.mcp_tools as mt

# 1) Fix for Colab: send the GitHub MCP server's log to a file
_log = open(os.path.join(tempfile.gettempdir(), "github_mcp.log"), "a")
_orig_init = mt.GitHubMcpToolset.__init__
def _patched_init(self, *a, **k):
    k.setdefault("errlog", _log)
    _orig_init(self, *a, **k)
mt.GitHubMcpToolset.__init__ = _patched_init
mt.mcp_tools[:] = mt._build_mcp_tools()
runners.clear(); sessions.clear()          # rebuild the agents with the fix

# 2) Retry automatically when Gemini is busy (503 / 429)
logging.getLogger("google_adk").setLevel(logging.CRITICAL)   # hide long tracebacks
_chat_once = chat
async def chat(text, mode="full", tries=4):
    for i in range(tries):
        try:
            return await _chat_once(text, mode)
        except Exception as e:
            busy = any(s in str(e) for s in ("503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED"))
            if busy and i < tries - 1:
                wait = 20 * (i + 1)
                print(f"   ⏳ Gemini is busy, retrying in {wait}s…")
                await asyncio.sleep(wait)
            else:
                print(f"   ❌ {type(e).__name__}: {str(e)[:200]}")
                return
print("✅ Patched: GitHub MCP fixed + automatic retry")

✅ Patched: GitHub MCP fixed + automatic retry


In [8]:
await chat("What meetings do I have tomorrow?")

🧑 What meetings do I have tomorrow?


/usr/local/lib/python3.13/dist-packages/google/adk/features/_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.BASE_AUTHENTICATED_TOOL is enabled.
  check_feature_enabled()


   🔧 get_current_datetime({})
   ↳ get_current_datetime → success
   🔧 list_upcoming_events({"start_date": "2026-10-12", "end_date": "2026-10-12"})
   ↳ list_upcoming_events → success | No events found in that range.
🤖 You have no meetings scheduled for tomorrow (Monday, October 12, 2026).
   📊 prompt tokens per model call: [9638, 9803, 9964]  (mode=full)



In [9]:
await chat("Find a free 1-hour slot next week")

🧑 Find a free 1-hour slot next week
   ⏳ Gemini is busy, retrying in 20s…
🧑 Find a free 1-hour slot next week
   🔧 find_available_slots({"end_date": "2026-10-16", "start_date": "2026-10-12", "duration_minutes": 60})
   ↳ find_available_slots → success
🤖 Here are some available 1-hour slots next week:

- Mon Oct 12, 9:00 AM – 10:00 AM
- Mon Oct 12, 10:00 AM – 11:00 AM
- Mon Oct 12, 11:00 AM – 12:00 PM
- Mon Oct 12, 1:00 PM – 2:00 PM
- Mon Oct 12, 2:00 PM – 3:00 PM

Would you like to schedule a meeting during one of these times, or look at a different day?
   📊 prompt tokens per model call: [10082, 10988]  (mode=full)



In [10]:
await chat("Am I free tomorrow at 3pm?")

🧑 Am I free tomorrow at 3pm?
   ⏳ Gemini is busy, retrying in 20s…
🧑 Am I free tomorrow at 3pm?
   🔧 check_conflicts({"start_time": "2026-10-12 15:00", "duration_minutes": 60})
   ↳ check_conflicts → success | That time is free.
🤖 Yes, you are free tomorrow (Monday, October 12) from 3:00 PM to 4:00 PM.
   📊 prompt tokens per model call: [11209, 11452]  (mode=full)



Create a test event. The agent should **ask for confirmation first**; the next cell answers *yes*.

In [11]:
await chat('Schedule "Assignment 2 test" tomorrow at 3pm for 30 minutes')

🧑 Schedule "Assignment 2 test" tomorrow at 3pm for 30 minutes
🤖 I'll schedule **"Assignment 2 test"** on **Monday, October 12, 2026, from 3:00 PM to 3:30 PM** (no guests). 

Shall I go ahead?
   📊 prompt tokens per model call: [11513]  (mode=full)



In [12]:
await chat("Yes, go ahead")

🧑 Yes, go ahead
   ⏳ Gemini is busy, retrying in 20s…
🧑 Yes, go ahead
   🔧 create_event({"summary": "Assignment 2 test", "end_time": "2026-10-12 15:30", "duration_minutes": 30, "start_time": "2026-1…)
   ↳ create_event → success | Event created.
🤖 "Assignment 2 test" has been scheduled:

- Mon Oct 12, 3:00-3:30 PM - Assignment 2 test
   📊 prompt tokens per model call: [11770, 12107]  (mode=full)



In [13]:
await chat('Move "Assignment 2 test" to 4pm')

🧑 Move "Assignment 2 test" to 4pm
   ⏳ Gemini is busy, retrying in 20s…
🧑 Move "Assignment 2 test" to 4pm
   ⏳ Gemini is busy, retrying in 40s…
🧑 Move "Assignment 2 test" to 4pm
   ⏳ Gemini is busy, retrying in 60s…
🧑 Move "Assignment 2 test" to 4pm
   🔧 check_conflicts({"duration_minutes": 30, "start_time": "2026-10-12 16:00"})
   ↳ check_conflicts → success | That time is free.
🤖 I'll reschedule **"Assignment 2 test"** to **Monday, October 12, 2026, from 4:00 PM to 4:30 PM** (no guests).

Shall I go ahead?
   📊 prompt tokens per model call: [12247, 12876]  (mode=full)



In [16]:
await chat("Yes")

🧑 Yes
   ⏳ Gemini is busy, retrying in 20s…
🧑 Yes
   ⏳ Gemini is busy, retrying in 40s…
🧑 Yes
   ⏳ Gemini is busy, retrying in 60s…
🧑 Yes
   ❌ _ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'me


Error handling check (should explain the problem, not crash):

In [17]:
await chat("Schedule a meeting on 2027-01-01 at 10am")

🧑 Schedule a meeting on 2027-01-01 at 10am
   ⏳ Gemini is busy, retrying in 20s…
🧑 Schedule a meeting on 2027-01-01 at 10am
   ⏳ Gemini is busy, retrying in 40s…
🧑 Schedule a meeting on 2027-01-01 at 10am
   ⏳ Gemini is busy, retrying in 60s…
🧑 Schedule a meeting on 2027-01-01 at 10am
   ❌ _ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'me


## 8. GitHub (MCP) tests

In [ ]:
await chat("List my GitHub repositories")

In [ ]:
await chat("Show open issues in octocat/Hello-World")

In [ ]:
await chat("Show me the README from octocat/Hello-World")

## 9. Bonus: tool search (defer_loading)
Same questions in `search` mode. Watch for `search_github_tools` being called first, and compare the prompt tokens with section 8.

In [ ]:
await chat("List my GitHub repositories", mode="search")

In [ ]:
await chat("Show me the README from octocat/Hello-World", mode="search")

Side-by-side numbers for the reflection (real Gemini prompt tokens):

In [ ]:
import scripts.compare_context_tokens as cmp
await cmp.main()

## 10. Clean up the test event (optional)

In [ ]:
from tools.auth import get_calendar_service
svc = get_calendar_service()
found = list_upcoming_events(search_text="Assignment 2 test", max_results=5)
for e in found.get("events", []):
    svc.events().delete(calendarId="primary", eventId=e["event_id"]).execute()
    print("🗑️ deleted:", e["summary"], e["start"])
print("done")

---
⚠️ **Privacy:** the outputs above show your real calendar events. If you save this notebook to a **public** repo, use *Edit → Clear all outputs* first, or keep only test events visible.